# Fabric Defect Detection — Google Colab

**Hướng dẫn:**
1. Đảm bảo Runtime → Change runtime type → **T4 GPU**
2. Chạy từng cell theo thứ tự
3. Cell cuối tạo link Gradio public để demo

**Lần đầu chạy:** Cần upload 2 file zip lên Google Drive (xem Cell 3)

## Cell 1 — Kiểm tra GPU

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Thư mục lưu project trên Drive
DRIVE_DIR = '/content/drive/MyDrive/defect_detection'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted → {DRIVE_DIR}')

## Cell 3 — Upload code + data lên Drive

**Chạy script này trên máy local TRƯỚC** để tạo 2 file zip:
```cmd
cd D:\Projects\defect_detection
python scripts\pack_for_colab.py
```

Sau đó upload 2 file vào Google Drive thư mục `defect_detection/`:
- `code.zip` (~2 MB)
- `data_tsfabric_T1.zip` (~312 MB)

**Nếu đã upload rồi, bỏ qua cell này.**

In [ ]:
import os, zipfile

PROJECT_DIR = '/content/defect_detection'
CODE_ZIP    = f'{DRIVE_DIR}/code.zip'
DATA_ZIP    = f'{DRIVE_DIR}/data_tsfabric_T1.zip'

# Giải nén code
if not os.path.exists(PROJECT_DIR):
    print('Giải nén code...')
    with zipfile.ZipFile(CODE_ZIP, 'r') as z:
        z.extractall('/content')
    print('✓ Code OK')
else:
    print('✓ Code đã có')

# Giải nén data
data_dir = f'{PROJECT_DIR}/data/pass/train/tsfabric_T1'
if not os.path.exists(data_dir):
    print('Giải nén data (~312 MB)...')
    with zipfile.ZipFile(DATA_ZIP, 'r') as z:
        z.extractall(PROJECT_DIR)
    print('✓ Data OK')
else:
    print('✓ Data đã có')

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

## Cell 4 — Cài dependencies

In [ ]:
import sys
sys.path.insert(0, '/content/defect_detection')
os.chdir('/content/defect_detection')

# SAM2 từ source (bắt buộc)
if not os.path.exists('/content/segment-anything-2'):
    !git clone -q https://github.com/facebookresearch/segment-anything-2 /content/segment-anything-2
    %cd /content/segment-anything-2
    !pip install -e . -q
    %cd /content/defect_detection
    print('✓ SAM2 installed')
else:
    print('✓ SAM2 already installed')

# Các package còn lại
!pip install -q gradio>=4.0 faiss-cpu rich peft transformers accelerate bitsandbytes
print('✓ All packages installed')

## Cell 5 — Download model weights

In [ ]:
import os

# SAM2 weights
sam2_path = 'weights/sam2/sam2_hiera_large.pt'
os.makedirs('weights/sam2', exist_ok=True)
if not os.path.exists(sam2_path):
    print('Downloading SAM2 weights (~900 MB)...')
    !wget -q --show-progress https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt -O {sam2_path}
    print('✓ SAM2 weights downloaded')
else:
    print('✓ SAM2 weights already exist')

# InternVL2-1B sẽ tự download từ HuggingFace khi cần
# DINOv2 sẽ tự download từ torch.hub khi cần
print('✓ Ready')

## Cell 6 — Copy checkpoints từ Drive (nếu đã train)

In [ ]:
import shutil, os

os.makedirs('outputs/checkpoints', exist_ok=True)
CKPT_DRIVE = f'{DRIVE_DIR}/checkpoints'

if os.path.exists(CKPT_DRIVE):
    for f in os.listdir(CKPT_DRIVE):
        src = f'{CKPT_DRIVE}/{f}'
        dst = f'outputs/checkpoints/{f}'
        if not os.path.exists(dst):
            if os.path.isdir(src):
                shutil.copytree(src, dst)
            else:
                shutil.copy2(src, dst)
            print(f'  Copied: {f}')
    print('✓ Checkpoints loaded from Drive')
else:
    print('Chưa có checkpoint trên Drive — cần train trước (Cell 7)')

!ls outputs/checkpoints/

## Cell 7 — Train (chạy nếu chưa có checkpoint)

Bỏ qua nếu đã có đủ checkpoint trong `outputs/checkpoints/`

In [ ]:
import os
os.environ['PYTHONPATH'] = '/content/defect_detection'

ckpt_dir = 'outputs/checkpoints'
stage1_ok = os.path.exists(f'{ckpt_dir}/stage1_tsfabric_T1_bank.npy')
stage2_ok = os.path.exists(f'{ckpt_dir}/stage2_tsfabric_T1_lora.pt')
stage3_ok = os.path.exists(f'{ckpt_dir}/stage3_tsfabric_T1_lora')

print(f'Stage 1: {"✓" if stage1_ok else "✗ cần train"}')
print(f'Stage 2: {"✓" if stage2_ok else "✗ cần train"}')
print(f'Stage 3: {"✓" if stage3_ok else "✗ cần train"}')

if not stage1_ok:
    print('\nTraining Stage 1...')
    !python -m stage1_anomaly.train --category tsfabric_T1

if not stage2_ok:
    print('\nTraining Stage 2...')
    !python -m stage2_seg.train --category tsfabric_T1

if not stage3_ok:
    print('\nTraining Stage 3...')
    !python -m stage3_vlm.lora_train --category tsfabric_T1

## Cell 8 — Lưu checkpoint về Drive (sau khi train)

In [ ]:
import shutil, os

CKPT_DRIVE = f'{DRIVE_DIR}/checkpoints'
os.makedirs(CKPT_DRIVE, exist_ok=True)

for f in os.listdir('outputs/checkpoints'):
    src = f'outputs/checkpoints/{f}'
    dst = f'{CKPT_DRIVE}/{f}'
    if not os.path.exists(dst):
        if os.path.isdir(src):
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        print(f'  Saved: {f}')

print(f'✓ Checkpoints saved → {CKPT_DRIVE}')
!ls {CKPT_DRIVE}

## Cell 9 — Chạy Gradio UI (public link)

Sau khi chạy cell này, copy link dạng `https://xxxx.gradio.live` để demo.

In [ ]:
import sys, os
sys.path.insert(0, '/content/defect_detection')
os.chdir('/content/defect_detection')
os.environ['PYTHONPATH'] = '/content/defect_detection'

# Chạy Gradio với public share link
!python app.py --share --category tsfabric_T1

---
## (Optional) Test inference 1 ảnh không cần UI

In [ ]:
import sys, os
sys.path.insert(0, '/content/defect_detection')
os.chdir('/content/defect_detection')

TEST_IMAGE = 'data/fail/train/tsfabric_T1/000003.jpeg'
!python inference/run_pipeline.py --image {TEST_IMAGE} --category tsfabric_T1